# 분류 이론
저번 세션에서 배운 분류 이론을 복습해봅시다.

노션에 있는 분류 이론 파트를 참고해도 좋아요! 

내용을 다시 읽어보면서 정리한다는 느낌으로 문제를 풀어주세요.

##
***머신러닝이 무엇인지 설명하세요.***

답: 기계가 모델을 학습해 새로운 데이터를 예측하거나 의사결정할 수 있도록 하는 기술

##
***머신러닝에는 지도학습과 비지도학습이 있습니다. 지도학습과 비지도학습의 차이를 설명해주세요.***

답: 지도학습은 기계에게 문제, 정답을 보여주고 학습시키는 것, 비지도학습은 기계에게 정답을 보여주지 않고 학습시킨다는 차이가 있다.

##
***대표적인 지도학습 모델로는 회귀와 분류가 있습니다. 회귀와 분류의 차이를 설명해주세요.***

답: 예측의 결과로 나올 수 있는 값이 연속형이면 회귀를, 범주형이면 분류를 진행함.

##
***이진분류와 다중분류의 차이를 설명해주세요.***

답: 분류의 결과로 나올 수 있는 결과값의 Class가 오직 True/False 두가지이면 이진, 나올 수 있는 결과값의 Class가 3개 이상이면 다중 분류라 말한다.

##
***세션에서 공부한 네 종류의 분류 모델을 간략히 설명해주세요.***

답 : 분류 모델에는 **로지스틱 회귀 모델, 의사결정나무 모델, SVM 모델, kNN 모델**이 있다.<br>
로지스틱 회귀 모델은 독립 변수의 선형 회귀에 로지스틱 함수를 적용해 0-1사이의 값이 출력되게 변환해주는 것을 말한다.<br>
의사결정나무 모델은 조건에 따라 데이터를 분류하여, 데이터가 순수한 라벨의 값으로 구성되도록 분류를 반복하는 방법이다.<br>
SVM은 Support Vector Model의 약자로, 클래스를 분류할 수 있는 최적의 선(초평면)을 찾는 방법이다.<br>
kNN은 k-Nearest Neighbor의 약자로, 데이가가 데이터로부터 가까운 k개의 집합의 레이블을 참고해 분류하는 방법이다.

# 분류 실습: 탑승한 항구를 예측하는 다중 분류 모델 만들기
저번 세션 시간에, 타이타닉 데이터셋을 이용해 **생존 여부(Survived)** 를 예측하는 분류 모델을 만들었었습니다.

그 모델은 **Survived/Not Survived** 를 예측하는 **이진 분류 모델** 이었습니다.

이번 과제에서는, **탑승한 항구(Embarked)** 를 예측하는 **다중 분류 모델** 을 만들어 볼 것입니다.

(탑승한 항구 컬럼의 값은 S, C, Q로 나뉘기 때문에, '이진 분류'가 아닌 '다중 분류'를 사용합니다.)

📌 어떤 사람에 대한 정보가 주어졌을 때, **그 사람이 어떤 항구에서 탑승했는지 예측하는 모델**만들어봅시다.
- 주어지는 정보: 생존 여부, 좌석 등급, 성별, 나이 등의 정보 (모델의 독립 변수)
- 예측하고자 하는 정보: 탑승한 항구 (모델의 종속 변수)

[ 변수 설명 ]

- PassengerId : 각 승객의 고유 번호

- Survived : 생존 여부(종속 변수)

        0 = 사망
        1 = 생존
 
- Pclass : 객실 등급 - 승객의 사회적, 경제적 지위

        1st = Upper
        2nd = Middle
        3rd = Lower

- Name : 이름

- Sex : 성별

- Age : 나이

- SibSp : 동반한 Sibling(형제자매)와 Spouse(배우자)의 수

- Parch : 동반한 Parent(부모) Child(자식)의 수

- Ticket : 티켓의 고유넘버

- Fare : 티켓의 요금

- Cabin : 객실 번호

- Embarked : 승선한 항

## 데이터 읽기 및 전처리

In [28]:
# seaborn을 sns, pandas를 pd, numpy를 np로 import해주세요
import seaborn as sns
import pandas as pd
import numpy as np

# 타이타닉 데이터셋을 불러와서 df에 저장해주세요
df = pd.read_csv("./titanic.csv")

In [29]:
# initial 컬럼을 만들고 일시적으로 값을 0으로 초기화
df['Initial'] = 0

for index, row in df.iterrows():
    initial_search = row['Name'].split(',')[1].split('.')[0].strip() # Name 컬럼에서 .(dot)을 기준으로 알파벳 문자열 추출
    df.at[index, 'Initial'] = initial_search

/var/folders/99/6tz9j7ss6j57ztqqvqn2vrv40000gn/T/ipykernel_16695/2791606022.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Mr' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, 'Initial'] = initial_search


In [30]:
# 유추 가능한 값들로 대체하고, 흔하지 않은 Initial들은 Other로 대체하겠습니다.
df['Initial'].replace([
    'Mlle', 'Mme', 'Ms', 'Dr', 'Major', 'Lady', 'Countess', 'Jonkheer', 'Col',
    'Rev', 'Capt', 'Sir', 'Don','the Countess' 
], [
    'Miss', 'Miss', 'Miss', 'Mr', 'Mr', 'Mrs', 'Mrs', 'Other', 'Other',
    'Other', 'Mr', 'Mr', 'Mr', 'Other'
],
    inplace=True)

/var/folders/99/6tz9j7ss6j57ztqqvqn2vrv40000gn/T/ipykernel_16695/2395359540.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Initial'].replace([


In [31]:
# 결측값을 Initial별 평균값으로 대체
df.loc[(df['Age'].isnull()) & (df.Initial == 'Mr'), 'Age'] = 33
df.loc[(df['Age'].isnull()) & (df.Initial == 'Mrs'), 'Age'] = 36
df.loc[(df['Age'].isnull()) & (df.Initial == 'Master'), 'Age'] = 5
df.loc[(df['Age'].isnull()) & (df.Initial == 'Miss'), 'Age'] = 22
df.loc[(df['Age'].isnull()) & (df.Initial == 'Other'), 'Age'] = 46

In [32]:
# Embarked 열의 결측값을 제거해주세요
df.dropna(subset=['Embarked'],inplace=True)

# 'Cabin', 'Name', 'PassengerId', 'Ticket' 열은 분석에서 제외하겠습니다.
df.drop(['Cabin', 'Name', 'PassengerId', 'Ticket'],axis=1,inplace=True)

In [33]:
# 나중에 사용하기 위해 원본 데이터프레임을 저장해둡니다.
df_org = df.copy()

# SibSp 행과 Parch 행을 이용해 Relatives 열을 만듭니다.
df['Relatives'] = df["SibSp"] + df["Parch"]

In [34]:
# Sex 열 인코딩
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
# Age 열을 10년 단위로 나누어 인코딩
df['Age'] = (df['Age'] // 10).astype(int)
# Fare 열을 9분위로 구간화하고 인코딩
df['Fare'] = pd.qcut(df['Fare'], q=9, labels=range(9))
# Embarked 열 인코딩
df['Embarked'] = df['Embarked'].map({'S': 1, 'C': 2, 'Q': 3})
# Initial 열 인코딩
initial_mapping = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Other':4}
df['Initial'] = df['Initial'].map(initial_mapping)

## 다중 분류 모델 target과 feature 정의

앞서 말했지만, 우리의 목표는 어떤 사람의 정보가 주어졌을 때 그 사람이 탑승한 항구를 예측하는 것입니다. 모델의 target과 feature가 무엇인지 정의해주세요.

- feature(예측을 위해 주어지는 정보 = 독립 변수): ***(답) 한 사람에 대한 정보 = Survived, Pclass, Sex, Age, SibSp, Parch, Fare, Initial, Relatives***
- target(예측하고자 하는 값 = 종속 변수): ***(답) Embarked***

이 target과 feature에 따라 생존 여부를 예측하는 함수 predict_survival를 정의합니다.

이 함수에 model과 독립 변수들을 넣어주면, 승선한 항 예측 값과 확률을 반환합니다.

- 함수의 input: model, scaler, 독립 변수(pclass, sex, age, sibsp, parch, fare, survived, initial)
- 함수의 output: 승선한 항구, 확률

In [35]:
# predict_survival 함수 정의
def predict_survival(model, scaler, survived, pclass, sex, age, sibsp, parch, fare, initial):
    # 입력된 데이터를 데이터프레임으로 변환합니다.
    # 위에서 피처엔지니어링한 방식대로(모델의 input으로 적절하게) input 값을 변환해줍니다. 
    input_data = pd.DataFrame({
        'Survived': [survived],
        'Pclass': [pclass],
        'Sex': [0 if sex == 'male' else 1],  # 성별을 인코딩합니다.
        'Age': [age // 10],  # 나이를 10년 단위로 나눕니다.
        'SibSp': [sibsp],
        'Parch': [parch],
        'Fare': [fare],  # 요금을 일단 그대로 둡니다.
        'Initial': [0 if initial == 'Mr' else (1 if initial == 'Miss' else (2 if initial == 'Mrs' else (3 if initial == 'Master' else 4)))],  # 호칭을 인코딩합니다.
        'Relatives': [sibsp + parch],
    })
    
    # 'Fare' 값을 qcut으로 생성된 bins를 사용해 범주화합니다.
    fare_bins = pd.qcut(df_org['Fare'], 9, retbins=True)[1]  # pd.qcut을 사용해 요금 구간을 얻습니다.
    input_data['Fare'] = pd.cut(input_data['Fare'], bins=fare_bins, labels=False, include_lowest=True)


    # 입력 데이터를 스케일링합니다.
    input_data_scaled = scaler.transform(input_data)
    
    # 예측을 수행합니다.
    prediction = model.predict(input_data_scaled)
    prediction_proba = model.predict_proba(input_data_scaled)

    # 예측 결과를 반환합니다.
    result = "S" if prediction == 1 else ("C" if prediction == 2 else "Q")
    probability = prediction_proba[0][int(prediction)]  # 예측된 클래스의 확률을 반환합니다.
    
    return result, probability

## target과 feature 분리
데이터 셋을 target과 feature를 분리합니다.
예측하고자 하는 값인 target과 예측하기 위해 주어진 값인 feature를 각각 변수에 담습니다.

- target = 종속 변수 (변수명=y): 'Embarked'
- feature = 독립 변수 (변수명=X): 'Pclass', 'Sex', 'Age', 'Sibsp', 'Parch', 'Fare', `Survived', 'Initial'

[ 참고 ]

drop() 함수는 drop한 후의 데이터프레임을 반환합니다.
drop('drop하고자 하는 열', axis=1)

위에서 주어진 참고 코드를 바탕으로 아래 주석에 맞게 코드를 작성해주세요.

In [36]:
# 변수 X에 feature(= 'Embarked' 열을 drop한 데이터프레임)를 담아주세요.
X=df.drop(['Embarked'],axis=1,inplace=False)

In [37]:
X

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Initial,Relatives
0,0,3,0,2,1,0,0,0,1
1,1,1,1,3,1,0,7,2,1
2,1,3,1,2,0,0,2,1,0
3,1,1,1,3,1,0,7,2,1
4,0,3,0,3,0,0,2,0,0
...,...,...,...,...,...,...,...,...,...
886,0,2,0,2,0,0,3,4,0
887,1,1,1,1,0,0,6,1,0
888,0,3,1,2,1,2,5,1,3
889,1,1,0,2,0,0,6,0,0


In [38]:
# 변수 y에 target(= 'Embarked' 열)을 담아주세요.
y = df['Embarked']

In [39]:
y

0      1
1      2
2      1
3      1
4      1
      ..
886    1
887    1
888    1
889    2
890    3
Name: Embarked, Length: 889, dtype: int64

## 데이터 셋을 훈련 세트와 테스트 세트로 나누기
- 훈련 세트 -> 모델을 학습시키는 데 사용됩니다.
- 테스트 세트 -> 완성된 모델을 평가하는 데 사용됩니다.

독립변수(X)와 종속변수(y)를 모두 훈련 세트와 테스트 세트로 나누어봅시다.

[ 변수명 ]

- 독립변수 훈련 세트: X_train
- 독립변수 테스트 세트: X_test
- 종속변수 훈련 세트: y_train
- 종속변수 테스트 세트: y_test

[ 참고 ]

tran_test_split 함수는 변수를 훈련 세트와 테스트 세트로 나누어 반환합니다.
~~~
train_test_split(독립변수, 종속변수, test_size=테스트 세트 크기, random_state=시드값)
~~~
-> 독립변수 훈련 세트, 독립변수 테스트 세트, 종속변수 훈련 세트, 종속변수 테스트 세트 반환

- test_size: 테스트 세트의 크기를 결정합니다. 예를 들어, 훈련 세트와 테스트 세트를 각각 70%와 30%로 하고 싶으면, test_size=0.3으로 하면 됩니다.
- random_state: 임의의 숫자로 설정된 시드로, 어떤 숫자를 사용해도 상관없습니다.

위에서 주어진 참고 코드를 바탕으로 아래 주석에 맞게 코드를 작성해주세요.

In [42]:
# sklearn.model_selectio의 train_test_split 함수를 import 해주세요.
from sklearn.model_selection import train_test_split

# tran_test_split 함수를 이용하여 독립변수(X)와 종속변수(y)를 각각 훈련 세트와 테스트 세트로 나누어주세요. 각 훈련 세트와 테스트 세트의 변수 명은 아래와 같습니다.
# 독립변수 훈련 세트: X_train, 독립변수 테스트 세트: X_test, 종속변수 훈련 세트: y_train, 종속변수 테스트 세트: y_test
# 단, random_state는 42로 합니다.
X_train,X_test,y_train,y_test = train_test_split(X, y, train_size=0.2, random_state=42)

# feature에 scaler 적용
MinMaxScaler를 이용해서 모든 feature(독립변수, X)를 스케일링 해줍니다. 

[ 변수명 ]
- 스케일링된 독립변수 훈련 세트: X_train_scaled
- 스케일링된 독립변수 테스트 세트: X_test_scaled

[ 참고 ]

1. MinMaxScaler 함수는 MinMaxScaler를 반환합니다.
~~~
MinMaxScaler()
~~~
2. fit_transform 함수는 훈련 데이터의 스케일링을 수행하고, 스케일링된 데이터를 반환합니다.
~~~
스케일러.fit_transform(훈련 데이터)
~~~
3. transform 함수는 테스트 데이터의 스케일링을 수행하고, 스케일링된 데이터를 반환합니다.
~~~
스케일러.transform(테스트 데이터)
~~~

cf) 훈련 데이터에 대해서는 fit과 transform을, 테스트 데이터에 대해서는 transform만을 수행합니다.

위에서 주어진 참고 코드를 바탕으로 아래 주석에 맞게 코드를 작성해주세요.

In [44]:
# sklearn.preprocessing에서 MinMaxScaler를 import 해옵니다.
from sklearn.preprocessing import MinMaxScaler

# scaler라는 변수를 MinMaxScaler 함수로 선언합니다.
scaler = MinMaxScaler()

# X_train_scaled라는 변수에 스케일링된 독립변수 훈련 세트를 저장합니다.
X_train_sclaed = scaler.fit_transform(X_train)

# X_test_scaled라는 변수에 스케일링된 독립변수 테스트 세트를 저장합니다.
X_test_scaled = scaler.transform(X_test)

## 모델 생성 및 평가
위에서 만든 함수에 input으로 들어갈 model을 만듭니다. 총 4개의 다중 분류 모델을 만들어볼 것입니다.
1. 로지스틱 회귀
2. 의사 결정 나무
3. 서포트벡터머신(SVM)
4. kNN

또한, 생성한 모델들을 다음과 같은 방식으로 평가합니다.
1. accuracy
2. 분류 보고서

cf) 혼동 행렬은 이진분류에 대해 만들어지므로, 이번 실습에서 사용하지 않습니다.

### 로지스틱 회귀
로지스틱 회귀 이진 분류 모델을 생성합니다.
sklearn 라이브러리는 로지스틱 회귀 모델(LogisticRegression)을 제공합니다.

[ 참고 ]
로지스틱 회귀 모델 생성 함수
~~~
LogisticRegression()

- penalty: 어떤 방식의 규제를 적용할지 선택합니다.

- C: 규제의 강도를 조절하는 파라미터입니다.

- solver: 모델의 최적 가중치를 찾기 위해 사용하는 최적화 알고리즘을 선택합니다.
~~~
위에서 주어진 참고 코드를 바탕으로 아래 주석에 맞게 코드를 작성해주세요.

In [45]:
# sklearn.linear_model에서 LogisticRegression 함수를 import 해오세요.
from sklearn.linear_model import LogisticRegression

# lr_model을 변수명으로 해서 로지스틱회귀 모델 객체를 생성하세요.
lr_model = LogisticRegression()

이제 모델을 학습하고, 평가해보도록 하겠습니다.

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 모델 학습
lr_model.fit(X_train, y_train)

# 예측 결과 생성
lr_pred = lr_model.predict(X_test)

# 정확도 측정
lr_accuracy = accuracy_score(y_test, lr_pred)
print("로지스틱 회귀 모델의 정확도:", lr_accuracy)

# 분류 보고서 생성
lr_report = classification_report(y_test, lr_pred)
print(lr_report)

로지스틱 회귀 모델의 정확도: 0.7148876404494382
              precision    recall  f1-score   support

           1       0.74      0.95      0.83       515
           2       0.44      0.16      0.24       137
           3       0.00      0.00      0.00        60

    accuracy                           0.71       712
   macro avg       0.39      0.37      0.35       712
weighted avg       0.62      0.71      0.64       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


이제, 생성한 모델을 사용해보겠습니다.

2번에서 행성했던 predict_survival 함수를 사용하여 예측값과 확률을 구합니다.

아래는 어떤 사람에 대한 정보입니다. 생성한 lr 모델을 이용해서 이 사람이 생존할지 생존하지 못할지, 또 그 예측이 맞을 확률은 어느 정도인지 구하세요.

[ 정보 ]
- pclass: 2
- sex: 'female'
- age: 32 
- sibsp: 1
- parch: 2 
- fare: 60
- survived: 1
- initial: 'Mrs'


In [47]:
model = lr_model

result, probability = predict_survival(
    model, scaler, 
    survived=1,
    pclass=2, sex='female', age=32, 
    sibsp=1, parch=2, 
    fare=60, initial='Mrs'
)

result, probability

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/var/folders/99/6tz9j7ss6j57ztqqvqn2vrv40000gn/T/ipykernel_16695/1060574916.py:31: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  probability = prediction_proba[0][int(prediction)]  # 예측된 클래스의 확률을 반환합니다.


('S', np.float64(0.0779636470693563))

### 의사 결정 나무
이번에는 의사 결정 나무 모델을 만들어보겠습니다. 다른 과정은 모두 똑같고, 로지스틱 회귀 모델 대신 의사 결정 나무 모델을 사용합니다.

sklearn 라이브러리는 의사 결정 나무 모델(DecisionTreeClassifier)을 제공합니다.

[ 참고 ]
의사 결정 나무 모델 생성 함수
~~~
DecisionTreeClassifier(random_state=시드값)

- random_state: 임의의 숫자로 설정된 시드로, 어떤 숫자를 사용해도 상관없습니다.

- max_depth: 트리의 최대 깊이를 제한합니다. 이 값을 작게 설정할수록 모델이 단순해집니다.

- min_samples_split: 노드를 분할하기 위해 필요한 최소한의 데이터(샘플) 개수를 지정합니다. 이 값을 높게 설정하면 트리의 성장을 억제하여 과적합을 방지하는 효과가 있습니다.

- min_samples_leaf: 분할 후, 리프 노드가 가져야 하는 최소한의 데이터(샘플) 개수를 지정합니다. min_samples_split과 비슷하지만, 분할 이후의 조건을 검사합니다. 예를 들어, 어떤 노드를 분할했을 때 자식 노드 중 하나의 데이터 개수가 이 값보다 작아진다면 해당 분할은 수행되지 않습니다. 이 역시 모델을 부드럽게(smoothing) 하고 과적합을 막는 역할을 합니다.

- ccp_alpha:(Cost-Complexity Pruning) 비용 복잡도 가지치기(Pruning)에 사용되는 파라미터입니다. 값이 클수록 더 많은 가지가 잘려나가 트리가 단순해집니다.

- criterion: 노드를 분할할 때 어떤 기준으로 불순도(Impurity)를 측정할지 결정합니다. 불순도는 한 노드에 여러 클래스의 데이터가 얼마나 섞여 있는지를 나타내는 지표입니다. 'gini' 아니면 'entropy'를 선택할 수 있습니다.

- max_features: 최적의 분할을 찾기 위해 고려할 피처(변수)의 최대 개수를 지정합니다. 매 분할마다 모든 피처를 고려하는 대신, 무작위로 선택된 일부 피처 중에서만 최적의 분할 기준을 찾습니다. 이는 트리가 특정 피처에 과도하게 의존하는 것을 막아주며, 특히 피처가 매우 많을 때 과적합 방지 및 훈련 속도 향상에 도움이 됩니다.
~~~

위에서 주어진 참고 코드를 바탕으로 아래 주석에 맞게 코드를 작성해주세요.

In [48]:
# sklearn.tree에서 DecisionTreeClassifier를 import 해오세요.
from sklearn.tree import DecisionTreeClassifier

# tree_model이라는 변수에 의사 결정 나무 모델을 생성해주세요. 단, random_state는 42로 합니다.
tree_model = DecisionTreeClassifier(random_state=42)

이제 모델을 학습, 평가하고 사용해보겠습니다.

In [49]:
# 모델 학습
tree_model.fit(X_train, y_train)

# 예측 결과 생성
tree_pred = tree_model.predict(X_test)

# 정확도 측정
tree_accuracy = accuracy_score(y_test, tree_pred)
print("Decision Tree 모델의 정확도:", tree_accuracy)

# 분류 보고서 생성
tree_report = classification_report(y_test, tree_pred)
print(tree_report)

model = tree_model

result, probability = predict_survival(
    model, scaler, 
    survived=1,
    pclass=2, sex='female', age=32, 
    sibsp=1, parch=2, 
    fare=60, initial='Mrs'
)

result, probability

Decision Tree 모델의 정확도: 0.6952247191011236
              precision    recall  f1-score   support

           1       0.78      0.86      0.82       515
           2       0.37      0.22      0.27       137
           3       0.38      0.40      0.39        60

    accuracy                           0.70       712
   macro avg       0.51      0.49      0.49       712
weighted avg       0.67      0.70      0.68       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/var/folders/99/6tz9j7ss6j57ztqqvqn2vrv40000gn/T/ipykernel_16695/1060574916.py:31: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  probability = prediction_proba[0][int(prediction)]  # 예측된 클래스의 확률을 반환합니다.


('S', np.float64(0.0))

### 서포트 벡터 머신 (SVM)
서포트 벡터 머신 역시 같은 방식으로 모델을 만듭니다.
sklearn 라이브러리는 서포트 벡터 머신 분류기(SVC=Support Vector Classification)을 제공합니다.

[ 참고 ]
서포트 벡터 머신 모델 생성 함수
~~~
SVC(random_state=시드값, probability=True/False)

- random_state: 임의의 숫자로 설정된 시드로, 어떤 숫자를 사용해도 상관없습니다.

- probability: 모델이 클래스의 확률을 제공할 수 있도록 하는 옵션입니다.

- C: 마진의 너비와 오류 데이터(Margin Violation)를 얼마나 허용할지를 결정하는 파라미터입니다.

- kernel: 데이터를 어떤 공간에서 바라볼지 결정합니다.

- gamma: 비선형 커널에서만 의미가 있는 파라미터입니다. 하나의 데이터 샘플이 경계선에 영향을 미치는 범위를 결정합니다.
~~~

위에서 주어진 참고 코드를 바탕으로 아래 주석에 맞게 코드를 작성해주세요.

In [50]:
# sklearn.svm에서 SVC를 import 해오세요.
from sklearn.svm import SVC

# svm_model 단, random_state는 42, probability=True로 합니다.
svm_model = SVC(random_state=42, probability=True)

이제 모델을 학습, 평가하고 사용해보겠습니다.

In [51]:
# 모델 훈련
svm_model.fit(X_train,y_train)

# SVM 예측 결과 생성
svm_pred = svm_model.predict(X_test)

# 정확도 측정
svm_accuracy = accuracy_score(y_test,svm_pred)
print("SVM 모델의 정확도:", svm_accuracy)

# 분류 보고서 생성
svm_report = classification_report(y_test, svm_pred)
print(svm_report)

# 생성한 모델로 예측값과 확률 도출
model = svm_model

result, probability = predict_survival(
    model, scaler, 
    survived=1,
    pclass=2, sex='female', age=32, 
    sibsp=1, parch=2, 
    fare=60, initial='Mrs'
)

result, probability

SVM 모델의 정확도: 0.7289325842696629
              precision    recall  f1-score   support

           1       0.74      0.97      0.84       515
           2       0.55      0.12      0.20       137
           3       0.00      0.00      0.00        60

    accuracy                           0.73       712
   macro avg       0.43      0.37      0.35       712
weighted avg       0.64      0.73      0.65       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python

('S', np.float64(0.09287550322476965))

### kNN
kNN 역시 같은 방식으로 모델을 만듭니다.
sklearn 라이브러리는 kNN 모델(KNeighborsClassifier)을 제공합니다.

[ 참고 ]
kNN 모델 생성 함수
~~~
KNeighborsClassifier(n_neighbors=이웃 수)

- n_neighbors: 고려할 최근접 이웃의 개수입니다.

- weights: k개의 이웃을 찾았을 때, 그들의 의견을 어떤 가중치로 반영할지 결정합니다.

- metric: 데이터 포인트 사이의 '거리'를 어떤 방식으로 측정할지 결정합니다.
~~~

위에서 주어진 참고 코드를 바탕으로 아래 주석에 맞게 코드를 작성해주세요.

In [52]:
# sklearn.neighbors에서 KNeighborsClassifier를 import 해오세요.
from sklearn.neighbors import KNeighborsClassifier

# knn_model이라는 변수에 kNN 모델을 생성해주세요. 단, n_neighbors=5로 합니다.
knn_model = KNeighborsClassifier(n_neighbors=5)

이제 모델을 학습, 평가하고 사용해보겠습니다.

In [54]:
# 모델 학습
knn_model.fit(X_train,y_train)

# 예측 수행
knn_pred = knn_model.predict(X_test)

# 정확도 측정
knn_accuracy = accuracy_score(y_test,knn_pred)
print("kNN 모델의 정확도 : ", knn_accuracy)

# 분류 보고서 생성
knn_report = classification_report(y_test, knn_pred)
print(knn_report)

# 생성한 모델로 예측값과 확률 도출
model = knn_model

result, probability = predict_survival(
    model, scaler, 
    survived=1,
    pclass=2, sex='female', age=32, 
    sibsp=1, parch=2, 
    fare=60, initial='Mrs'
)


result, probability

kNN 모델의 정확도 :  0.7134831460674157
              precision    recall  f1-score   support

           1       0.78      0.86      0.82       515
           2       0.45      0.27      0.34       137
           3       0.44      0.45      0.45        60

    accuracy                           0.71       712
   macro avg       0.56      0.53      0.53       712
weighted avg       0.69      0.71      0.70       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/var/folders/99/6tz9j7ss6j57ztqqvqn2vrv40000gn/T/ipykernel_16695/1060574916.py:31: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  probability = prediction_proba[0][int(prediction)]  # 예측된 클래스의 확률을 반환합니다.


('S', np.float64(0.0))

## 하이퍼파라미터 튜닝

마지막으로 하이퍼파라미터 튜닝을 통해 모델의 성능을 개선하는 방법을 알아봅시다.

의사결정나무에 Grid Search와 Random Search를 적용해봅시다!

[ 참고 ]
의사결정나무 모델 주요 하이퍼파라미터
~~~
DecisionTreeClassifier()

- max_depth: 트리의 최대 깊이를 제한합니다. 이 값을 작게 설정할수록 모델이 단순해집니다.

- min_samples_split: 노드를 분할하기 위해 필요한 최소한의 데이터(샘플) 개수를 지정합니다. 이 값을 높게 설정하면 트리의 성장을 억제하여 과적합을 방지하는 효과가 있습니다.

- min_samples_leaf: 분할 후, 리프 노드가 가져야 하는 최소한의 데이터(샘플) 개수를 지정합니다. min_samples_split과 비슷하지만, 분할 이후의 조건을 검사합니다. 예를 들어, 어떤 노드를 분할했을 때 자식 노드 중 하나의 데이터 개수가 이 값보다 작아진다면 해당 분할은 수행되지 않습니다. 이 역시 모델을 부드럽게(smoothing) 하고 과적합을 막는 역할을 합니다.

- ccp_alpha:(Cost-Complexity Pruning) 비용 복잡도 가지치기(Pruning)에 사용되는 파라미터입니다. 값이 클수록 더 많은 가지가 잘려나가 트리가 단순해집니다.

- criterion: 노드를 분할할 때 어떤 기준으로 불순도(Impurity)를 측정할지 결정합니다. 불순도는 한 노드에 여러 클래스의 데이터가 얼마나 섞여 있는지를 나타내는 지표입니다. 'gini' 아니면 'entropy'를 선택할 수 있습니다.

- max_features: 최적의 분할을 찾기 위해 고려할 피처(변수)의 최대 개수를 지정합니다. 매 분할마다 모든 피처를 고려하는 대신, 무작위로 선택된 일부 피처 중에서만 최적의 분할 기준을 찾습니다. 이는 트리가 특정 피처에 과도하게 의존하는 것을 막아주며, 특히 피처가 매우 많을 때 과적합 방지 및 훈련 속도 향상에 도움이 됩니다.
~~~


💡 하이퍼파라미터 최적화 방법
- Grid Search: 정해진 범위에서 Hyperparameter를 모두 순회
- Random Search: 정해진 범위에서 Hyperparameter를 무작위로 탐색
- Bayesian Optimization: 사전 정보를 바탕으로 Hyperparameter 값을 확률적으로 추정하며 탐색

### Grid Search
정해진 범위에서 Hyperparameter를 모두 순회하며 가장 좋은 성능을 내는 값을 찾는 기법
- **장점**: 범위가 넓고 step이 작을수록 꼼꼼하게 전 범위를 탐색하니 최적해를 **정확히 찾을 수 있다**.
- **단점**: 시간이 너무 오래 걸린다.
- **적용**: 넓은 범위, 큰 step을 활용해 범위를 좁힌다.

[ 참고 ]
Grid Search 주요 하이퍼파라미터
~~~
GridSearchCV()

- estimator: 튜닝할 모델 객체를 지정합니다.

- param_grid: 테스트할 하이퍼파라미터들의 목록을 사전(dictionary) 형태로 전달합니다.

- cv: 교차 검증(Cross-Validation)을 어떻게 수행할지 지정합니다.

- scoring: 최적의 하이퍼파라미터를 선택하기 위한 평가 지표를 지정합니다.

- n_jobs: 튜닝을 수행할 때 사용할 CPU 코어의 개수를 지정합니다.

- verbose: 튜닝 과정에서 출력되는 메시지의 양을 조절합니다.

In [58]:
from sklearn.model_selection import GridSearchCV

# Decision Tree 모델 생성
tree_model = DecisionTreeClassifier()

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_grid = {
    'max_depth': [3, 5, 7, 10],         # 트리의 최대 깊이
    'min_samples_split': [2, 5, 10],    # 노드를 나누기 위한 최소 샘플 수
    'min_samples_leaf': [1, 3, 5]       # 리프 노드가 되기 위한 최소 샘플 수
}

# GridSearchCV 객체 생성 (cv: 5겹 교차검증)
# n_jobs=-1 은 사용 가능한 모든 CPU 코어를 사용하여 학습 속도를 높입니다.
grid_search = GridSearchCV(estimator=tree_model, param_grid=param_grid, 
                           cv=5, verbose=1, n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
grid_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", grid_search.best_params_)
print(f"최고 교차검증 정확도: {grid_search.best_score_:.4f}")

# Grid Search가 찾은 최적의 모델을 저장
best_tree_model = grid_search.best_estimator_

Fitting 5 folds for each of 36 candidates, totalling 180 fits
최적 하이퍼파라미터: {'max_depth': 10, 'min_samples_leaf': 3, 'min_samples_split': 2}
최고 교차검증 정확도: 0.7857


In [64]:
# 최적 모델로 예측 결과 생성
tree_pred = best_tree_model.predict(X_test)

# 정확도 측정
tree_accuracy = accuracy_score(y_test, tree_pred)
print("\n튜닝된 Decision Tree 모델의 정확도:", tree_accuracy)

# 분류 보고서 생성
tree_report = classification_report(y_test, tree_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(tree_report)


튜닝된 Decision Tree 모델의 정확도: 0.7134831460674157

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.78      0.86      0.82       515
           2       0.46      0.30      0.36       137
           3       0.42      0.42      0.42        60

    accuracy                           0.71       712
   macro avg       0.56      0.52      0.53       712
weighted avg       0.69      0.71      0.70       712



In [65]:
# 최적화된 모델인 'best_tree_model'을 사용합니다.
model = best_tree_model

result, probability = predict_survival(
    model, scaler, 
    survived=1,
    pclass=2, sex='female', age=32, 
    sibsp=1, parch=2, 
    fare=60, initial='Mrs'
)

print("\n[새로운 데이터에 대한 예측 결과]")
print("예측 결과:", result)
print("생존 확률:", probability)


[새로운 데이터에 대한 예측 결과]
예측 결과: S
생존 확률: 0.0


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/var/folders/99/6tz9j7ss6j57ztqqvqn2vrv40000gn/T/ipykernel_16695/1060574916.py:31: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  probability = prediction_proba[0][int(prediction)]  # 예측된 클래스의 확률을 반환합니다.


### Random Search
정해진 범위에서 Hyperparameter를 **무작위**로 탐색해 가장 좋은 성능을 내는 값을 찾는 기법
- **장점**: 속도가 Grid Search보다 빠르다.
- **단점**: 무작위라는 한계 때문에 **정확도가 떨어진다**. 따라서 Grid Search나 Bayesian Optimization에 비해 사용 빈도가 적다.

[ 참고 ]
Grid Search 주요 하이퍼파라미터
~~~
RandomizedSearchCV()

- estimator: 튜닝할 모델 객체를 지정합니다.

- param_distributions: 탐색할 하이퍼파라미터의 분포 또는 목록을 사전(dictionary) 형태로 지정합니다.

- n_iter: 지정된 param_distributions에서 몇 개의 하이퍼파라미터 조합을 무작위로 추출하여 테스트할지 그 횟수를 결정합니다.

- random_state: 결과의 재현성(reproducibility)을 위한 파라미터입니다.

- cv: 교차 검증 분할 개수

- scoring: 최적 모델 선택을 위한 평가 지표

- n_jobs: 사용할 CPU 코어 수

- verbose: 진행 과정 출력 메시지 양

In [66]:
from sklearn.model_selection import RandomizedSearchCV

# 기본 Decision Tree 모델 생성
tree_model = DecisionTreeClassifier()

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_dist = {
    'max_depth': [3, 5, 7, 10, 15, 20, None], # None은 깊이 제한 없음을 의미
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 6, 8],
    'criterion': ['gini', 'entropy']
}

# RandomizedSearchCV 객체 생성
# n_iter: 시도할 파라미터 조합의 수 (많을수록 좋은 조합을 찾을 확률이 높지만, 시간이 오래 걸림)
# cv: 5겹 교차검증
# n_jobs=-1: 사용 가능한 모든 CPU 코어를 사용하여 학습 속도 향상
random_search = RandomizedSearchCV(estimator=tree_model, 
                                   param_distributions=param_dist,
                                   n_iter=100, # 100개의 파라미터 조합을 무작위로 테스트합니다.
                                   cv=5, 
                                   verbose=1, 
                                   random_state=42, # 결과를 재현하기 위해 random_state 설정
                                   n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
random_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", random_search.best_params_)
print(f"최고 교차검증 정확도: {random_search.best_score_:.4f}")

# Random Search가 찾은 최적의 모델을 저장
best_tree_model = random_search.best_estimator_

Fitting 5 folds for each of 100 candidates, totalling 500 fits


/opt/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPi

최적 하이퍼파라미터: {'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 7, 'criterion': 'entropy'}
최고 교차검증 정확도: 0.8027


In [67]:
# 최적 모델로 예측 결과 생성
tree_pred = best_tree_model.predict(X_test)

# 정확도 측정
tree_accuracy = accuracy_score(y_test, tree_pred)
print("\n튜닝된 Decision Tree 모델의 정확도:", tree_accuracy)

# 분류 보고서 생성
tree_report = classification_report(y_test, tree_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(tree_report)


튜닝된 Decision Tree 모델의 정확도: 0.723314606741573

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.79      0.87      0.83       515
           2       0.51      0.31      0.39       137
           3       0.42      0.42      0.42        60

    accuracy                           0.72       712
   macro avg       0.57      0.53      0.54       712
weighted avg       0.70      0.72      0.71       712



In [68]:
# 3. 튜닝된 최적 모델로 새로운 데이터 예측
model = best_tree_model

result, probability = predict_survival(
    model, scaler, 
    survived=1,
    pclass=2, sex='female', age=32, 
    sibsp=1, parch=2, 
    fare=60, initial='Mrs'
)

print("\n[새로운 데이터에 대한 예측 결과]")
print("예측 결과:", result)
print("생존 확률:", probability)


[새로운 데이터에 대한 예측 결과]
예측 결과: S
생존 확률: 0.0


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/var/folders/99/6tz9j7ss6j57ztqqvqn2vrv40000gn/T/ipykernel_16695/1060574916.py:31: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  probability = prediction_proba[0][int(prediction)]  # 예측된 클래스의 확률을 반환합니다.


## 끝으로...

이번 세션에서 배운 모델들(로지스틱 회귀, 의사결정나무, SVM, kNN)의 하이퍼파라미터를 튜닝하여 가장 성능이 좋은 모델을 만들어주세요!

정확도가 가장 높은 1등에게는 소정의 상품이 지급될 예정입니다!

위에서 의사결정 나무 모델을 Grid와 Random으로 튜닝해보았으니, 로지스틱 회귀, SVM, kNN을 순차적으로 튜닝하여 성능을 비교해보겠다.

In [72]:
!pip install bayesian-optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bayesian-optimization]


## 로지스틱 회귀

### Grid Search

In [133]:
from sklearn.model_selection import GridSearchCV

# 기본 로지스틱회귀 모델 객체 생성
lr_model = LogisticRegression()

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'], # 가장 범용적인 l2 규제 사용
    'solver': ['lbfgs', 'liblinear']
}

# GridSearchCV 객체 생성 (cv: 5겹 교차검증)
grid_search = GridSearchCV(estimator=lr_model, param_grid=param_grid, 
                           cv=5, verbose=1, n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
grid_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", grid_search.best_params_)
print(f"최고 교차검증 정확도: {grid_search.best_score_:.4f}")

# Grid Search가 찾은 최적의 모델을 저장
best_lr_model = grid_search.best_estimator_

Fitting 5 folds for each of 10 candidates, totalling 50 fits
최적 하이퍼파라미터: {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}
최고 교차검증 정확도: 0.7289


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

In [134]:
# 최적 모델로 예측 결과 생성
lr_pred = best_lr_model.predict(X_test)

# 정확도 측정
lr_accuracy = accuracy_score(y_test, lr_pred)
print("\n튜닝된 로지스틱 회귀 모델의 정확도:", lr_accuracy)

# 분류 보고서 생성
lr_report = classification_report(y_test, lr_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(lr_report)


튜닝된 Decision Tree 모델의 정확도: 0.723314606741573

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.72      1.00      0.84       515
           2       0.00      0.00      0.00       137
           3       0.00      0.00      0.00        60

    accuracy                           0.72       712
   macro avg       0.24      0.33      0.28       712
weighted avg       0.52      0.72      0.61       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Random Search

In [145]:
from sklearn.model_selection import RandomizedSearchCV

# 기본 로지스틱회귀 모델 객체 생성
lr_model = LogisticRegression()

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'], # 가장 범용적인 l2 규제 사용
    'solver': ['lbfgs', 'liblinear']
}

# RandomizedSearchCV 객체 생성
# n_iter: 시도할 파라미터 조합의 수 (많을수록 좋은 조합을 찾을 확률이 높지만, 시간이 오래 걸림)
# cv: 5겹 교차검증
# n_jobs=-1: 사용 가능한 모든 CPU 코어를 사용하여 학습 속도 향상
random_search = RandomizedSearchCV(estimator=lr_model, 
                                   param_distributions=param_grid,
                                   n_iter=100, # 100개의 파라미터 조합을 무작위로 테스트
                                   cv=5, 
                                   verbose=1, 
                                   random_state=42, # 결과를 재현하기 위해 random_state 설정
                                   n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
random_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", random_search.best_params_)
print(f"최고 교차검증 정확도: {random_search.best_score_:.4f}")

# Random Search가 찾은 최적의 모델을 저장
best_lr_model = random_search.best_estimator_

/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 10 is smaller than n_iter=100. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf inste

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda

최적 하이퍼파라미터: {'solver': 'lbfgs', 'penalty': 'l2', 'C': 0.01}
최고 교차검증 정확도: 0.7289


In [146]:
# 최적 모델로 예측 결과 생성
lr_pred = best_lr_model.predict(X_test)

# 정확도 측정
lr_accuracy = accuracy_score(y_test, lr_pred)
print("\n튜닝된 로지스틱 회귀 모델의 정확도:", lr_accuracy)

# 분류 보고서 생성
lr_report = classification_report(y_test, lr_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(lr_report)


튜닝된 로지스틱 회귀 모델의 정확도: 0.723314606741573

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.72      1.00      0.84       515
           2       0.00      0.00      0.00       137
           3       0.00      0.00      0.00        60

    accuracy                           0.72       712
   macro avg       0.24      0.33      0.28       712
weighted avg       0.52      0.72      0.61       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## SVM

### Grid Search

In [119]:
from sklearn.model_selection import GridSearchCV

# 기본 서포트 벡터 머신 모델 객체 생성
svm_model = SVC(random_state=42, probability=True)

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_grid = {
    'C': [1, 10, 100, 1000],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'sigmoid']
}

# GridSearchCV 객체 생성 (cv: 5겹 교차검증)
grid_search = GridSearchCV(estimator=svm_model, param_grid=param_grid, 
                           cv=5, verbose=1, n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
grid_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", grid_search.best_params_)
print(f"최고 교차검증 정확도: {grid_search.best_score_:.4f}")

# Grid Search가 찾은 최적의 모델을 저장
best_svm_model = grid_search.best_estimator_

Fitting 5 folds for each of 32 candidates, totalling 160 fits
최적 하이퍼파라미터: {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
최고 교차검증 정확도: 0.7970


In [120]:
# 최적 모델로 예측 결과 생성
svm_pred = best_svm_model.predict(X_test)

# 정확도 측정
svm_accuracy = accuracy_score(y_test, svm_pred)
print("\n튜닝된 SVM 모델의 정확도:", svm_accuracy)

# 분류 보고서 생성
svm_report = classification_report(y_test, svm_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(svm_report)


튜닝된 SVM 모델의 정확도: 0.7120786516853933

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.74      0.93      0.82       515
           2       0.43      0.22      0.29       137
           3       0.00      0.00      0.00        60

    accuracy                           0.71       712
   macro avg       0.39      0.38      0.37       712
weighted avg       0.62      0.71      0.65       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Random Search

In [124]:
from sklearn.model_selection import RandomizedSearchCV

# 기본 서포트 벡터 머신 모델 객체 생성
svm_model = SVC()

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_grid = {
    'C': [1, 10, 100, 1000],
    'gamma': [0.0001, 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'sigmoid']
}

# RandomizedSearchCV 객체 생성
# n_iter: 시도할 파라미터 조합의 수 (많을수록 좋은 조합을 찾을 확률이 높지만, 시간이 오래 걸림)
# cv: 5겹 교차검증
# n_jobs=-1: 사용 가능한 모든 CPU 코어를 사용하여 학습 속도 향상
random_search = RandomizedSearchCV(estimator=svm_model, 
                                   param_distributions=param_grid,
                                   n_iter=100, # 100개의 파라미터 조합을 무작위로 테스트
                                   cv=5, 
                                   verbose=1, 
                                   random_state=42, # 결과를 재현하기 위해 random_state 설정
                                   n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
random_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", random_search.best_params_)
print(f"최고 교차검증 정확도: {random_search.best_score_:.4f}")

# Random Search가 찾은 최적의 모델을 저장
best_svm_model = random_search.best_estimator_

/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 40 is smaller than n_iter=100. Running 40 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 40 candidates, totalling 200 fits
최적 하이퍼파라미터: {'kernel': 'rbf', 'gamma': 0.01, 'C': 100}
최고 교차검증 정확도: 0.7970


In [125]:
# 최적 모델로 예측 결과 생성
svm_pred = best_svm_model.predict(X_test)

# 정확도 측정
svm_accuracy = accuracy_score(y_test, svm_pred)
print("\n튜닝된 SVM 회귀 모델의 정확도:", svm_accuracy)

# 분류 보고서 생성
svm_report = classification_report(y_test, svm_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(svm_report)


튜닝된 SVM 회귀 모델의 정확도: 0.7120786516853933

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.74      0.93      0.82       515
           2       0.43      0.22      0.29       137
           3       0.00      0.00      0.00        60

    accuracy                           0.71       712
   macro avg       0.39      0.38      0.37       712
weighted avg       0.62      0.71      0.65       712



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## kNN

### Grid Search

In [126]:

from sklearn.model_selection import GridSearchCV

# 기본 knn 모델 생성
knn_model = KNeighborsClassifier(n_neighbors=5)

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_grid = {
    'n_neighbors': [3, 7, 13, 19, 25, 31, 35], ## 3부터 시작해서 제곱근인 27을 넘어 35 정도까지 홀수로 구성
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'p': [1, 2]
}

# GridSearchCV 객체 생성 (cv: 5겹 교차검증)
grid_search = GridSearchCV(estimator=knn_model, param_grid=param_grid, 
                           cv=5, verbose=1, n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
grid_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", grid_search.best_params_)
print(f"최고 교차검증 정확도: {grid_search.best_score_:.4f}")

# Grid Search가 찾은 최적의 모델을 저장
best_knn_model = grid_search.best_estimator_

Fitting 5 folds for each of 56 candidates, totalling 280 fits
최적 하이퍼파라미터: {'metric': 'manhattan', 'n_neighbors': 19, 'p': 1, 'weights': 'distance'}
최고 교차검증 정확도: 0.7856


In [127]:
# 최적 모델로 예측 결과 생성
knn_pred = best_knn_model.predict(X_test)

# 정확도 측정
knn_accuracy = accuracy_score(y_test, knn_pred)
print("\n튜닝된 kNN 모델의 정확도:", knn_accuracy)

# 분류 보고서 생성
knn_report = classification_report(y_test, knn_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(knn_report)


튜닝된 kNN 모델의 정확도: 0.7162921348314607

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.76      0.90      0.82       515
           2       0.43      0.18      0.26       137
           3       0.47      0.40      0.43        60

    accuracy                           0.72       712
   macro avg       0.56      0.49      0.50       712
weighted avg       0.68      0.72      0.68       712



### Random Search

In [137]:
from sklearn.model_selection import RandomizedSearchCV

# 기본 knn 모델 생성
knn_model = KNeighborsClassifier(n_neighbors=5)

# 튜닝할 하이퍼파라미터의 후보 값들 설정
param_grid = {
    'n_neighbors': [9, 11, 13, 15, 17, 19, 21, 23, 25],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'p': [1, 2]
    
}

# RandomizedSearchCV 객체 생성
# n_iter: 시도할 파라미터 조합의 수 (많을수록 좋은 조합을 찾을 확률이 높지만, 시간이 오래 걸림)
# cv: 5겹 교차검증
# n_jobs=-1: 사용 가능한 모든 CPU 코어를 사용하여 학습 속도 향상
random_search = RandomizedSearchCV(estimator=knn_model, 
                                   param_distributions=param_grid,
                                   n_iter=100, # 100개의 파라미터 조합을 무작위로 테스트
                                   cv=5, 
                                   verbose=1, 
                                   random_state=42, # 결과를 재현하기 위해 random_state 설정
                                   n_jobs=-1)

# 최적의 하이퍼파라미터를 찾기 위해 모델 학습
random_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 그때의 최고 점수 출력
print("최적 하이퍼파라미터:", random_search.best_params_)
print(f"최고 교차검증 정확도: {random_search.best_score_:.4f}")

# Random Search가 찾은 최적의 모델을 저장
best_knn_model = random_search.best_estimator_

/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 72 is smaller than n_iter=100. Running 72 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 72 candidates, totalling 360 fits
최적 하이퍼파라미터: {'weights': 'distance', 'p': 1, 'n_neighbors': 17, 'metric': 'euclidean'}
최고 교차검증 정확도: 0.7856


In [138]:
# 최적 모델로 예측 결과 생성
knn_pred = best_knn_model.predict(X_test)

# 정확도 측정
knn_accuracy = accuracy_score(y_test, knn_pred)
print("\n튜닝된 kNN 회귀 모델의 정확도:", knn_accuracy)

# 분류 보고서 생성
knn_report = classification_report(y_test, knn_pred)
print("\n[튜닝된 모델의 분류 보고서]")
print(knn_report)


튜닝된 kNN 회귀 모델의 정확도: 0.7275280898876404

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.78      0.89      0.83       515
           2       0.50      0.25      0.33       137
           3       0.47      0.40      0.43        60

    accuracy                           0.73       712
   macro avg       0.58      0.51      0.53       712
weighted avg       0.70      0.73      0.70       712



## 1차 결론

여기까지 Grid Search와 Random Search를 통해 의사결정, 로지스틱, SVM, kNN 모델을 튜닝해봤을 때, <br>가장 좋은 성능의 모델은 **Random Search로 튜닝한 kNN 모델**인 것으로 보임 (정확도 0.72752)<br><br>
다만, Bayesian Optimazition을 통해 더 효율적으로 최적의 하이퍼파라미터를 찾을 수 있다하니, <br>아래부터는 Gemini의 도움을 받아서,, 최적의 하이퍼 파라미터를 찾아봄..

### Bayes Opt - 의사결정나무

In [161]:
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. 목적 함수 정의
def tree_cv(max_depth, min_samples_split, min_samples_leaf, criterion_idx):
    # criterion 매핑: 0이면 'gini', 1이면 'entropy'
    criteria = ['gini', 'entropy']
    curr_criterion = criteria[int(criterion_idx)]
    
    model = DecisionTreeClassifier(
        max_depth=int(max_depth),
        min_samples_split=int(min_samples_split),
        min_samples_leaf=int(min_samples_leaf),
        criterion=curr_criterion,
        random_state=42
    )
    
    # 5-겹 교차검증 평균 점수 반환
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    return scores.mean()

# 2. 탐색 범위(Bounds) 설정
# max_depth의 경우 None(제한 없음)은 바예지안에서 표현하기 어려우므로, 
# 충분히 큰 숫자인 20이나 30을 상한선으로 잡아줍니다.
pbounds = {
    'max_depth': (3, 20),
    'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10),
    'criterion_idx': (0, 1.99)  # 0: gini, 1: entropy
}

# 3. 옵티마이저 객체 생성 및 실행
optimizer = BayesianOptimization(
    f=tree_cv,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

# 학습 시작 (무작위 5번 후, 똑똑한 탐색 20번)
optimizer.maximize(init_points=5, n_iter=20)

# 4. 최적 파라미터 추출 및 모델 재학습
params = optimizer.max['params']
criteria = ['gini', 'entropy']

best_tree_model = DecisionTreeClassifier(
    max_depth=int(params['max_depth']),
    min_samples_split=int(params['min_samples_split']),
    min_samples_leaf=int(params['min_samples_leaf']),
    criterion=criteria[int(params['criterion_idx'])],
    random_state=42
)

best_tree_model.fit(X_train, y_train)

|   iter    |  target   | max_depth | min_sa... | min_sa... | criter... |
-------------------------------------------------------------------------
| 1         | 0.7515873 | 9.3671820 | 19.112857 | 7.5879454 | 1.1913303 |
| 2         | 0.7631746 | 5.6523168 | 4.8079013 | 1.5227525 | 1.7236905 |
| 3         | 0.7801587 | 13.218955 | 14.745306 | 1.1852604 | 1.9301206 |
| 4         | 0.7573015 | 17.151524 | 5.8221039 | 2.6364247 | 0.3649749 |
| 5         | 0.7236507 | 8.1721181 | 11.445615 | 4.8875051 | 0.5795459 |
| 6         | 0.7180952 | 16.347991 | 4.7386619 | 4.4688176 | 0.9463618 |
| 7         | 0.7744444 | 10.662304 | 13.679611 | 2.3827386 | 1.5110060 |
| 8         | 0.7798412 | 10.812090 | 5.6425471 | 1.8147907 | 0.6269281 |
| 9         | 0.7801587 | 12.021822 | 14.267351 | 1.7265511 | 1.7383398 |
| 10        | 0.7971428 | 13.344696 | 11.805853 | 1.0       | 1.8015146 |
| 11        | 0.7855555 | 15.476079 | 11.059443 | 1.0       | 0.0       |
| 12        | 0.7857142 | 13.107434 | 

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",13
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",11
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [162]:
# 5. 결과 출력 및 검증
print("\n최적 하이퍼파라미터:", params)
print(f"최고 교차검증 정확도: {optimizer.max['target']:.4f}")

tree_pred = best_tree_model.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_pred)
print(f"\n튜닝된 의사결정나무 회귀 모델의 정확도: {lr_accuracy:.4f}")
print("\n[튜닝된 모델의 분류 보고서]")
print(classification_report(y_test, tree_pred))


최적 하이퍼파라미터: {'max_depth': np.float64(13.344696515356985), 'min_samples_split': np.float64(11.805853516270034), 'min_samples_leaf': np.float64(1.0), 'criterion_idx': np.float64(1.8015146634664279)}
최고 교차검증 정확도: 0.7971

튜닝된 의사결정나무 회귀 모델의 정확도: 0.7317

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.78      0.88      0.83       515
           2       0.51      0.26      0.35       137
           3       0.42      0.42      0.42        60

    accuracy                           0.72       712
   macro avg       0.57      0.52      0.53       712
weighted avg       0.70      0.72      0.70       712



### Bayes Opt - 로지스틱

In [157]:
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score

# 1. 목적 함수 정의 (multi_class 옵션 삭제)
def lr_cv(C):
    model = LogisticRegression(
        C=C, 
        solver='lbfgs',      # 다중 분류를 알아서 잘 처리하는 lbfgs 고정
        penalty='l2', 
        max_iter=1000, 
        random_state=42
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    return scores.mean()

# 2. 탐색 범위 설정 (C만 집중 공략!)
pbounds = {'C': (0.01, 100)}

# 3. 옵티마이저 실행
optimizer = BayesianOptimization(f=lr_cv, pbounds=pbounds, random_state=42, verbose=2)
optimizer.maximize(init_points=5, n_iter=20)

# 4. 최적 모델 재학습
params = optimizer.max['params']

best_lr_model = LogisticRegression(
    C=params['C'],
    solver='lbfgs',          # 목적 함수와 동일하게 설정
    penalty='l2',
    max_iter=1000,
    random_state=42
)

best_lr_model.fit(X_train, y_train)

|   iter    |  target   |     C     |
-------------------------------------
| 1         | 0.6952380 | 37.460266 |
| 2         | 0.6952380 | 95.071923 |
| 3         | 0.6952380 | 73.202074 |
| 4         | 0.6952380 | 59.869861 |


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

| 5         | 0.6952380 | 15.610303 |
| 6         | 0.7288888 | 0.0111435 |
| 7         | 0.7174603 | 0.8533522 |
| 8         | 0.7063492 | 3.9152273 |


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

| 9         | 0.7007936 | 6.5695786 |
| 10        | 0.6952380 | 48.667324 |
| 11        | 0.7287301 | 0.2624206 |
| 12        | 0.6952380 | 84.132009 |
| 13        | 0.6952380 | 26.535307 |


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

| 23        | 0.6952380 | 20.754947 |
| 24        | 0.6952380 | 31.999845 |


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

| 25        | 0.7288888 | 0.0798756 |


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",np.float64(0....4151485088625)
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems

In [158]:

# 5. 결과 출력 (소수점 4자리)
print("\n최적 하이퍼파라미터:", params)
print(f"최고 교차검증 정확도: {optimizer.max['target']:.4f}")

lr_pred = best_lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_pred)

print(f"\n튜닝된 로지스틱 회귀 모델의 정확도: {lr_accuracy:.4f}")
print("\n[튜닝된 모델의 분류 보고서]")
print(classification_report(y_test, lr_pred, digits=4))


최적 하이퍼파라미터: {'C': np.float64(0.13054151485088625)}
최고 교차검증 정확도: 0.7402

튜닝된 로지스틱 회귀 모델의 정확도: 0.7317

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1     0.7400    0.9728    0.8406       515
           2     0.5882    0.1460    0.2339       137
           3     0.0000    0.0000    0.0000        60

    accuracy                         0.7317       712
   macro avg     0.4428    0.3729    0.3582       712
weighted avg     0.6485    0.7317    0.6530       712



### Bayes Opt - SVM

In [155]:
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. 목적 함수 정의
def svm_cv(C, gamma, kernel_idx):
    # kernel 매핑: 0이면 'rbf', 1이면 'sigmoid'
    kernels = ['rbf', 'sigmoid']
    curr_kernel = kernels[int(kernel_idx)]
    
    model = SVC(
        C=C,                        # C는 연속적인 실수이므로 그대로 사용
        gamma=gamma,                # gamma도 연속적인 실수이므로 그대로 사용
        kernel=curr_kernel,
        probability=True,           # 필요 시 True 설정
        random_state=42
    )
    
    # 5-겹 교차검증 평균 점수 반환
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    return scores.mean()

# 2. 탐색 범위(Bounds) 설정
# SVM의 C와 gamma는 보통 로그 스케일로 탐색하므로 범위를 넉넉하게 잡습니다.
pbounds = {
    'C': (1, 1000),               # 지윤 님의 후보군 [1, 1000] 반영
    'gamma': (0.0001, 1),         # 지윤 님의 후보군 [0.0001, 1] 반영
    'kernel_idx': (0, 1.99)       # 0: rbf, 1: sigmoid
}

# 3. 옵티마이저 객체 생성 및 실행
optimizer = BayesianOptimization(
    f=svm_cv,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

# 학습 실행 (초기 5번 무작위, 이후 20번 최적화 탐색)
optimizer.maximize(init_points=5, n_iter=20)

# 4. 최적 파라미터 추출 및 모델 재학습
params = optimizer.max['params']
kernels = ['rbf', 'sigmoid']

best_svm_model = SVC(
    C=params['C'],
    gamma=params['gamma'],
    kernel=kernels[int(params['kernel_idx'])],
    probability=True,
    random_state=42
)

best_svm_model.fit(X_train, y_train)

|   iter    |  target   |     C     |   gamma   | kernel... |
-------------------------------------------------------------
| 1         | 0.6836507 | 375.16557 | 0.9507192 | 1.4566679 |
| 2         | 0.7404761 | 599.05982 | 0.1561030 | 0.3104290 |
| 3         | 0.7006349 | 59.025528 | 0.8661895 | 1.1962188 |
| 4         | 0.5358730 | 708.36450 | 0.0206824 | 1.9301206 |
| 5         | 0.7574603 | 832.61019 | 0.2124178 | 0.3618316 |
| 6         | 0.5820634 | 833.39619 | 0.0713243 | 1.4552767 |
| 7         | 0.6665079 | 476.08317 | 0.5154414 | 1.5660835 |
| 8         | 0.6665079 | 154.48398 | 0.7593236 | 1.1866007 |
| 9         | 0.7122222 | 203.16136 | 0.0906232 | 0.6269281 |
| 10        | 0.7293650 | 519.48721 | 0.1211127 | 0.1560113 |
| 11        | 0.7796825 | 107.99447 | 0.5253397 | 0.3174889 |
| 12        | 0.6553968 | 335.77821 | 0.3461348 | 1.8885463 |
| 13        | 0.7001587 | 166.62957 | 0.2040333 | 1.5873067 |
| 14        | 0.6665079 | 627.86456 | 0.5461201 | 1.6272748 |
| 15    

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",np.float64(816.7807535716707)
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",np.float64(0.6009127356037481)
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [156]:
# 5. 결과 출력 및 검증
print("\n최적 하이퍼파라미터:", params)
print(f"최고 교차검증 정확도: {optimizer.max['target']:.4f}")

svm_pred = best_svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_pred)
print(f"\n튜닝된 svm 회귀 모델의 정확도: {svm_accuracy:.4f}")
print("\n[튜닝된 모델의 분류 보고서]")
print(classification_report(y_test, svm_pred))


최적 하이퍼파라미터: {'C': np.float64(816.7807535716707), 'gamma': np.float64(0.6009127356037481), 'kernel_idx': np.float64(0.6623797097461542)}
최고 교차검증 정확도: 0.7852

튜닝된 svm 회귀 모델의 정확도: 0.7079

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.76      0.88      0.82       515
           2       0.39      0.16      0.23       137
           3       0.44      0.45      0.45        60

    accuracy                           0.71       712
   macro avg       0.53      0.50      0.50       712
weighted avg       0.67      0.71      0.67       712



### Bayes Opt - kNN

In [153]:
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score

# 1. 목적 함수 정의
def knn_cv(n_neighbors, weights_idx, metric_idx, p):
    # 바예지안 옵티마이저는 실수 값을 던져주므로, 정수형/카테고리형으로 변환 필수!
    
    # weights 매핑: 0이면 'uniform', 1이면 'distance'
    weights_list = ['uniform', 'distance']
    curr_weights = weights_list[int(weights_idx)]
    
    # metric 매핑: 0이면 'euclidean', 1이면 'manhattan'
    metric_list = ['euclidean', 'manhattan']
    curr_metric = metric_list[int(metric_idx)]
    
    model = KNeighborsClassifier(
        n_neighbors=int(n_neighbors),
        weights=curr_weights,
        metric=curr_metric,
        p=int(p)
    )
    
    # 5겹 교차검증 평균 점수 반환
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    return scores.mean()

# 2. 탐색 범위 설정
# 범주형 파라미터는 인덱스 범위(0 ~ 1.99)로 설정하여 int 변환 시 0 또는 1이 되게 함
pbounds = {
    'n_neighbors': (3, 40),       # 지윤 님이 고려하신 3~35 범위를 넉넉히 포함
    'weights_idx': (0, 1.99),     # uniform, distance
    'metric_idx': (0, 1.99),      # euclidean, manhattan
    'p': (1, 2.99)                # 1(Manhattan), 2(Euclidean)
}

# 3. 옵티마이저 객체 생성 및 실행
optimizer = BayesianOptimization(
    f=knn_cv,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

optimizer.maximize(init_points=5, n_iter=20)

# 4. 최적 파라미터 추출 및 모델 재학습
params = optimizer.max['params']

# 매핑 다시 적용 (최종 모델 생성을 위해)
weights_list = ['uniform', 'distance']
metric_list = ['euclidean', 'manhattan']

best_knn_model = KNeighborsClassifier(
    n_neighbors=int(params['n_neighbors']),
    weights=weights_list[int(params['weights_idx'])],
    metric=metric_list[int(params['metric_idx'])],
    p=int(params['p'])
)

best_knn_model.fit(X_train, y_train)



|   iter    |  target   | n_neig... | weight... | metric... |     p     |
-------------------------------------------------------------------------
| 1         | 0.7855555 | 16.857984 | 1.8919214 | 1.4566679 | 2.1913303 |
| 2         | 0.7633333 | 8.7726896 | 0.3104290 | 0.1155863 | 2.7236905 |
| 3         | 0.7684126 | 25.241255 | 1.4090644 | 0.0409631 | 2.9301206 |
| 4         | 0.7288888 | 33.800377 | 0.4225548 | 0.3618316 | 1.3649749 |
| 5         | 0.7741269 | 14.256962 | 1.0442652 | 0.8595705 | 1.5795459 |
| 6         | 0.7344444 | 20.299480 | 0.0       | 1.99      | 1.0       |
| 7         | 0.7741269 | 15.900435 | 1.99      | 0.1645557 | 2.99      |
| 8         | 0.7519047 | 15.995712 | 0.1345394 | 1.99      | 2.99      |
| 9         | 0.7912698 | 16.111939 | 1.99      | 0.8100870 | 1.1407352 |
| 10        | 0.7855555 | 17.701728 | 1.99      | 0.0       | 1.0       |
| 11        | 0.7684126 | 3.0       | 1.99      | 1.99      | 1.0       |
| 12        | 0.7569841 | 40.0      | 

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",16
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",1
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [154]:

# 5. 결과 출력 및 검증
print("\n최적 하이퍼파라미터:", params)
print(f"최고 교차검증 정확도: {optimizer.max['target']:.4f}")

knn_pred = best_knn_model.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_pred)

print(f"\n튜닝된 knn 회귀 모델의 정확도: {knn_accuracy:.4f}")
print("\n[튜닝된 모델의 분류 보고서]")
print(classification_report(y_test, knn_pred))


최적 하이퍼파라미터: {'n_neighbors': np.float64(16.111939913278288), 'weights_idx': np.float64(1.99), 'metric_idx': np.float64(0.81008702580445), 'p': np.float64(1.140735292001798)}
최고 교차검증 정확도: 0.7913

튜닝된 knn 회귀 모델의 정확도: 0.7275

[튜닝된 모델의 분류 보고서]
              precision    recall  f1-score   support

           1       0.78      0.89      0.83       515
           2       0.50      0.26      0.34       137
           3       0.47      0.40      0.43        60

    accuracy                           0.73       712
   macro avg       0.58      0.52      0.54       712
weighted avg       0.70      0.73      0.70       712



## 2차 결론
Bayesian Optimization까지 진행해보았을 때, 가장 성능이 좋은 모델은 **Bayesian으로 튜닝된 의사결정나무 모델**이다.

- 최고 교차검증 정확도: 0.7971
- 튜닝된 의사결정나무 회귀 모델의 정확도: 0.7317